# Config 1 time-resolved: electron and ion TOF vs delay (multi-file)

Loads one or more config-1 time-resolved aggregates
(`compute_aggregates.py` with `MODE = "time_resolved"`) and produces
side-by-side GMD-normalised eTOF / ion-TOF maps versus stage delay
`z` — one column per file (panels wrap to multiple rows past 3
files).

The notebook follows the `delay_compare_runs.ipynb` convention:
specify a list of files in `FILES` (and an optional matching
`LABELS`), and every downstream cell loops over a `runs` dict so
adding or removing a file just adds or removes a column.

Figures produced:

1. Raw GMD-normalised spectrum vs delay (eTOF + ion TOF rows per
   file). If `DELAY_BASELINE` is set, the delay-averaged spectrum
   from that window is subtracted from every delay slice.
2. Same maps with a per-delay TOF-window baseline subtracted.
   Stacks on top of the optional delay-baseline subtraction.
3. **1D summary**: eTOF integrated over `ETOF_ROI` plotted against
   delay, with one line per file. Subtractions configured above
   are applied before the integration.
4. Electron-ion and ion-ion partial covariance averaged over a
   chosen delay window (one column per file), with an
   `OVERSUBTRACT` factor on the GMD common-mode correction.

`ETOF_DOWNBIN` / `ION_TOF_DOWNBIN` rebin the TOF axes via a matrix
transformation (a `(n_new, n_old)` matrix `M` of ones that sums
every `k` adjacent bins). The downbinning is applied to D, C and to
the second-moment matrices DtD / CtC / DtC / DtG / CtG used by the
covariance cell, so every figure sees a consistent TOF axis.

In [ ]:
import sys
from pathlib import Path

_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import config
from compute_aggregates import load_aggregates
%matplotlib inline

## Parameters

`FILES` is the list of config-1 time-resolved aggregate paths to
compare. `LABELS` is an optional matching list of short human
labels; when shorter than `FILES`, the missing entries fall back to
each file's stem.

`GMD_BIN` is a single integer applied to every file — every file
must have at least `GMD_BIN + 1` GMD bins.

`ETOF_BASELINE_TOF` / `ION_TOF_BASELINE_TOF` give the TOF window
(in 100 ps units, same as `tof_edges`) used for plot 2's per-delay
DC baseline. `(None, None)` skips the subtraction.

`DELAY_BASELINE` (stage z units) selects a delay window whose
averaged spectrum is subtracted from every delay slice. `None`
skips. `ETOF_ROI` is the TOF window summed in plot 3;
`(None, None)` integrates the full TOF range.

`ETOF_DOWNBIN` / `ION_TOF_DOWNBIN` are integer factors that rebin
each TOF axis by summing every `k` adjacent bins (implemented as a
`(n_new, n_old)` 0/1 matrix `M`). `k = 1` is a no-op. Any tail bins
that don't fit a full group are dropped.

`COV_DELAY_WINDOW` is the stage-z window over which the partial
covariance cell averages the per-bin matrices before computing
covariances. `OVERSUBTRACT` scales the GMD common-mode correction
in the partial covariance: `1.0` = canonical partial cov, `> 1` =
oversubtracted.

In [ ]:
# --- input --------------------------------------------------------
FILES = [
    config.COMBINED_DIR / "glycine_delay_scan_150C_272.0eV_aggregates_tr.h5",
]
# Optional human-readable labels (one per file). Missing entries
# fall back to the file stem.
LABELS = []

# --- GMD bin selection -------------------------------------------
GMD_BIN  = 1            # integer index, applied to every file

# --- TOF downbinning (matrix transformation) ---------------------
# Sum every k adjacent bins of the TOF axis. 1 = no downbinning.
ETOF_DOWNBIN    = 1
ION_TOF_DOWNBIN = 1

# --- TOF-window baseline ("two TOF bins") ------------------------
# Mean over this TOF window is subtracted from every TOF bin in the
# same delay slice. Units = 100 ps (matches tof_edges after any
# downbinning).
ETOF_BASELINE_TOF    = (None, None)   # e.g. (0.0, 500.0)
ION_TOF_BASELINE_TOF = (None, None)

# --- Delay-window baseline ---------------------------------------
# Mean spectrum over this delay window is subtracted from every
# delay slice. Set to None to skip. Units = stage z.
DELAY_BASELINE = (-5000, -1000)        # e.g. (-15000.0, -10000.0)

# --- eTOF ROI for plot 3 -----------------------------------------
# TOF window (100 ps units) integrated for the 1D delay trace.
ETOF_ROI = (None, None)                # e.g. (1000.0, 4000.0)

# --- Covariance plot ---------------------------------------------
# Stage-z window to average over before forming covariances.
# (None, None) -> full delay range.
COV_DELAY_WINDOW = (None, None)

# Oversubtraction factor for the GMD common-mode correction.
# 1.0 = canonical partial covariance. >1.0 = oversubtract.
OVERSUBTRACT = 1.0

# --- cosmetics ---------------------------------------------------
PCT_LOW, PCT_HIGH = 1.0, 99.0          # robust colour-scale limits
MAX_COLS          = 3                  # subplot wrap threshold

print(f"files to compare: {len(FILES)}")
for p in FILES:
    print(f"  {p}")

## Load and normalise

Each file is loaded and validated as a config-1 time-resolved
aggregate. For the chosen `GMD_BIN`:

1. The TOF axis is downbinned via a `(n_new, n_old)` matrix `M_e`
   (and `M_i` for ions) of ones that sums every `k` adjacent bins.
   Applied to the first moments (D, C) as `D @ M_eᵀ`, and stashed
   for the covariance cell which will apply them to the second
   moments as `M @ X @ Mᵀ`.
2. The downbinned per-shot spectrum is GMD-normalised
   (`D_db / G[:, None]`) so each map is in counts-per-shot per uJ.

`runs` is a dict keyed by label; every downstream cell iterates
over `runs.items()`.

In [ ]:
def _downbin_matrix(n_old, k):
    """(n_new, n_old) 0/1 matrix that sums every k consecutive bins."""
    if k <= 0:
        raise ValueError(f"downbin factor must be >= 1, got {k}")
    n_new = n_old // k
    if n_new == 0:
        raise ValueError(f"downbin factor {k} > n_old={n_old}")
    M = np.zeros((n_new, n_old), dtype=np.float64)
    for i in range(n_new):
        M[i, i * k:(i + 1) * k] = 1.0
    return M


def _downbin_edges(edges, k):
    """Pick every k-th edge so the new bins line up with M."""
    n_old = len(edges) - 1
    n_new = n_old // k
    return np.asarray(edges[0:(n_new + 1) * k:k], dtype=np.float64)


def _label_for(i, path):
    if i < len(LABELS) and LABELS[i]:
        return LABELS[i]
    return Path(path).stem


def load_and_prepare(path):
    agg = load_aggregates(path)
    if agg.config != 1:
        raise ValueError(
            f"{Path(path).name}: expected config=1, got config={agg.config}"
        )
    if agg.mode != "time_resolved":
        raise ValueError(
            f"{Path(path).name}: expected mode='time_resolved', "
            f"got mode={agg.mode!r}"
        )
    if not (0 <= GMD_BIN < agg.n_gmd_bins):
        raise ValueError(
            f"{Path(path).name}: GMD_BIN={GMD_BIN} out of range "
            f"[0, {agg.n_gmd_bins})"
        )

    M_e = _downbin_matrix(agg.n_tof,   ETOF_DOWNBIN)
    M_i = _downbin_matrix(agg.n_tof_i, ION_TOF_DOWNBIN)

    # D, C: shape (n_z, n_tof_*). Downbin via x @ Mᵀ along TOF axis.
    D_db = agg.D[GMD_BIN] @ M_e.T
    C_db = agg.C[GMD_BIN] @ M_i.T

    with np.errstate(invalid="ignore", divide="ignore"):
        D = D_db / agg.G[GMD_BIN][:, None]
        C = C_db / agg.G[GMD_BIN][:, None]

    tof_edges_db     = _downbin_edges(agg.tof_edges,     ETOF_DOWNBIN)
    ion_tof_edges_db = _downbin_edges(agg.ion_tof_edges, ION_TOF_DOWNBIN)

    gmd_label = (f"GMD [{agg.gmd_edges[GMD_BIN]:.3g}, "
                 f"{agg.gmd_edges[GMD_BIN + 1]:.3g}) uJ")
    return {
        "agg": agg,
        "D": D,
        "C": C,
        "tof_edges":     tof_edges_db,
        "ion_tof_edges": ion_tof_edges_db,
        "z_edges":       agg.z_edges,
        "gmd_label":     gmd_label,
        "n_tof":   D.shape[-1],
        "n_tof_i": C.shape[-1],
        "n_z":     agg.n_z_bins,
        "n_per_bin": agg.n_per_bin[GMD_BIN],
        "M_e": M_e,
        "M_i": M_i,
    }


runs = {_label_for(i, p): load_and_prepare(p) for i, p in enumerate(FILES)}

for label, run in runs.items():
    print(f"{label}:")
    print(f"  GMD       : {run['gmd_label']}")
    print(f"  n_z       : {run['n_z']}")
    print(f"  n_tof (e) : {run['n_tof']}  (downbin x{ETOF_DOWNBIN})")
    print(f"  n_tof (i) : {run['n_tof_i']}  (downbin x{ION_TOF_DOWNBIN})")
    print(f"  shots/bin : min={run['n_per_bin'].min()}, "
          f"max={run['n_per_bin'].max()}, "
          f"total={run['n_per_bin'].sum()}")

## Subtraction + plotting helpers

`_subtract_tof_baseline`: per-delay mean over a TOF window,
subtracted from every TOF bin in the same delay slice.

`_subtract_delay_baseline`: mean spectrum over a delay window,
subtracted from every delay slice. Pass `None` or `(None, None)` to
skip either.

`_plot_panel` paints one map (pcolormesh) into a supplied axis,
either a robust viridis scale (raw) or a symmetric diverging
RdBu_r / `TwoSlopeNorm` scale (after any subtraction).

`_file_grid` allocates a 2D axes grid for the multi-file panels:
each file contributes `n_metric_rows` row-stacked axes, columns
wrap at `MAX_COLS`, and leftover panels are hidden.

`_window_avg` performs a shot-weighted average of a per-(z) array
across a delay window — used by the partial covariance cell.

In [ ]:
def _window_indices(edges, lo, hi):
    """Index range [i_lo, i_hi) of bins whose centres fall in [lo, hi)."""
    cent = 0.5 * (edges[:-1] + edges[1:])
    if lo is None:
        lo = cent[0]
    if hi is None:
        hi = cent[-1] + np.finfo(float).eps
    sel = (cent >= lo) & (cent < hi)
    idx = np.where(sel)[0]
    if idx.size == 0:
        raise ValueError(
            f"empty window [{lo}, {hi}) for edges spanning "
            f"[{edges[0]}, {edges[-1]})"
        )
    return int(idx[0]), int(idx[-1] + 1)


def _subtract_tof_baseline(spec, tof_edges, tof_window):
    if tof_window is None or tof_window == (None, None):
        return spec
    lo, hi = tof_window
    if lo is None and hi is None:
        return spec
    i0, i1 = _window_indices(tof_edges, lo, hi)
    base = np.nanmean(spec[:, i0:i1], axis=1, keepdims=True)
    return spec - base


def _subtract_delay_baseline(spec, z_edges, delay_window):
    if delay_window is None:
        return spec
    lo, hi = delay_window
    if lo is None and hi is None:
        return spec
    j0, j1 = _window_indices(z_edges, lo, hi)
    base = np.nanmean(spec[j0:j1, :], axis=0, keepdims=True)
    return spec - base


def _robust_limits(arr, pct_lo, pct_hi, symmetric=False):
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return -1.0, 1.0
    if symmetric:
        lim = float(np.nanpercentile(np.abs(finite), pct_hi))
        if lim <= 0:
            lim = float(np.nanmax(np.abs(finite))) or 1.0
        return -lim, lim
    return (float(np.nanpercentile(finite, pct_lo)),
            float(np.nanpercentile(finite, pct_hi)))


def _plot_panel(ax, spec, x_edges, y_edges, title, xlabel, cbar_label,
                symmetric):
    if symmetric:
        vmin, vmax = _robust_limits(spec, PCT_LOW, PCT_HIGH, symmetric=True)
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        pcm = ax.pcolormesh(x_edges, y_edges, spec, cmap="RdBu_r",
                            norm=norm, shading="auto")
    else:
        vmin, vmax = _robust_limits(spec, PCT_LOW, PCT_HIGH)
        pcm = ax.pcolormesh(x_edges, y_edges, spec, cmap="viridis",
                            vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    cbar = ax.figure.colorbar(pcm, ax=ax)
    cbar.set_label(cbar_label)
    return pcm


def _tag(window, name):
    if window is None or window == (None, None):
        return f"no {name}"
    return f"{name} [{window[0]}, {window[1]})"


def _delay_tag():
    return (f", delay [{DELAY_BASELINE[0]}, {DELAY_BASELINE[1]})"
            if DELAY_BASELINE is not None else "")


def _window_avg(arr, n_per_bin, j0, j1):
    """Shot-weighted average of per-(z) means over delay slice [j0, j1).

    `arr` has shape (n_z, ...). Empty bins (n_per_bin == 0) get
    zero weight and their NaN entries are zeroed out so they don't
    contaminate the running sum.
    """
    w = np.asarray(n_per_bin[j0:j1], dtype=np.float64)
    total = float(w.sum())
    if total == 0:
        return np.full(arr.shape[1:], np.nan, dtype=np.float64)
    sl = np.where(np.isnan(arr[j0:j1]), 0.0, arr[j0:j1])
    return np.tensordot(w, sl, axes=([0], [0])) / total


def _file_grid(n_metric_rows, n_files, panel_size=(6.0, 4.0),
               max_cols=None, sharey="row"):
    """Allocate a (n_metric_rows-per-file, n_files-cols) axes grid that
    wraps to multiple file-rows past ``max_cols`` columns.

    Returns ``(fig, axes2d, ax_at, hide_unused)`` where ``axes2d`` is
    the raw 2D ndarray, ``ax_at(k, m)`` is the axis for file index
    ``k`` and metric row ``m``, and ``hide_unused()`` turns off the
    panels left empty by the wrap.
    """
    if max_cols is None:
        max_cols = MAX_COLS
    cols = min(max_cols, n_files)
    file_rows = (n_files + cols - 1) // cols
    fig, axes = plt.subplots(
        n_metric_rows * file_rows, cols,
        figsize=(panel_size[0] * cols,
                 panel_size[1] * n_metric_rows * file_rows),
        sharey=sharey, constrained_layout=True, squeeze=False,
    )

    def ax_at(k, m):
        fr, fc = divmod(k, cols)
        return axes[fr * n_metric_rows + m, fc]

    def hide_unused():
        for k in range(n_files, file_rows * cols):
            fr, fc = divmod(k, cols)
            for m in range(n_metric_rows):
                axes[fr * n_metric_rows + m, fc].axis("off")

    return fig, axes, ax_at, hide_unused

## Plot 1 — raw GMD-normalised spectrum vs delay

Top metric row per file: eTOF map. Bottom metric row: ion TOF map.
Files wrap to multiple subplot-rows past `MAX_COLS` columns. If
`DELAY_BASELINE` is set, the spectrum averaged over that delay
window is subtracted from every delay slice (diverging colormap).
Otherwise the raw GMD-normalised spectrum is shown (viridis).

In [ ]:
sym1 = DELAY_BASELINE is not None
suffix = "  - delay-baseline" if sym1 else ""
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(2, n_files,
                                           panel_size=(6, 4))
for k, (label, run) in enumerate(runs.items()):
    D_p1 = _subtract_delay_baseline(run["D"], run["z_edges"], DELAY_BASELINE)
    C_p1 = _subtract_delay_baseline(run["C"], run["z_edges"], DELAY_BASELINE)

    _plot_panel(ax_at(k, 0), D_p1, run["tof_edges"], run["z_edges"],
                f"{label}  -  eTOF{suffix}",
                "eTOF (100 ps)", "D / G (counts/shot/uJ)", sym1)
    _plot_panel(ax_at(k, 1), C_p1, run["ion_tof_edges"], run["z_edges"],
                f"{label}  -  ion TOF{suffix}",
                "ion TOF (100 ps)", "C / G (counts/shot/uJ)", sym1)

hide_unused()
for ax in axes[:, 0]:
    ax.set_ylabel("stage z (arb.)")
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"GMD-normalised TOF vs delay  -  {gmd_labels}")
plt.show()

## Plot 2 — TOF-window baseline subtracted

For each delay slice the mean of the spectrum over the chosen TOF
window is subtracted from every TOF bin. Use this to remove a
delay-dependent DC offset (dark / scatter floor). If
`DELAY_BASELINE` is also set, the delay-averaged spectrum is
subtracted on top.

In [ ]:
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(2, n_files,
                                           panel_size=(6, 4))
for k, (label, run) in enumerate(runs.items()):
    D_p2 = _subtract_tof_baseline(run["D"], run["tof_edges"],
                                  ETOF_BASELINE_TOF)
    C_p2 = _subtract_tof_baseline(run["C"], run["ion_tof_edges"],
                                  ION_TOF_BASELINE_TOF)
    D_p2 = _subtract_delay_baseline(D_p2, run["z_edges"], DELAY_BASELINE)
    C_p2 = _subtract_delay_baseline(C_p2, run["z_edges"], DELAY_BASELINE)

    _plot_panel(
        ax_at(k, 0), D_p2, run["tof_edges"], run["z_edges"],
        f"{label}  -  eTOF\n{_tag(ETOF_BASELINE_TOF, 'TOF baseline')}{_delay_tag()}",
        "eTOF (100 ps)", "(D - baseline) / G", symmetric=True,
    )
    _plot_panel(
        ax_at(k, 1), C_p2, run["ion_tof_edges"], run["z_edges"],
        f"{label}  -  ion TOF\n{_tag(ION_TOF_BASELINE_TOF, 'TOF baseline')}{_delay_tag()}",
        "ion TOF (100 ps)", "(C - baseline) / G", symmetric=True,
    )

hide_unused()
for ax in axes[:, 0]:
    ax.set_ylabel("stage z (arb.)")
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"TOF-baseline subtracted  -  {gmd_labels}")
plt.show()

## Plot 3 — eTOF ROI integrated vs delay (1D overlay)

For each file, sum the (possibly baseline-subtracted) eTOF spectrum
over `ETOF_ROI` along the TOF axis, giving a 1D curve of integrated
counts vs delay. All files are overlaid on a single panel; the
y-axis is the GMD-normalised sum in the ROI (units of D / G summed
over the ROI TOF bins).

In [ ]:
sym_roi = (ETOF_BASELINE_TOF != (None, None)) or (DELAY_BASELINE is not None)

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)

for label, run in runs.items():
    D_roi = _subtract_tof_baseline(run["D"], run["tof_edges"],
                                   ETOF_BASELINE_TOF)
    D_roi = _subtract_delay_baseline(D_roi, run["z_edges"], DELAY_BASELINE)

    roi_lo, roi_hi = ETOF_ROI
    if roi_lo is None and roi_hi is None:
        i0, i1 = 0, run["n_tof"]
    else:
        i0, i1 = _window_indices(run["tof_edges"], roi_lo, roi_hi)
    print(f"{label}: eTOF ROI bins [{i0}, {i1}) -> "
          f"[{run['tof_edges'][i0]:.1f}, {run['tof_edges'][i1]:.1f}) 100 ps")

    integrated = D_roi[:, i0:i1].sum(axis=1)
    z_cent = 0.5 * (run["z_edges"][:-1] + run["z_edges"][1:])
    ax.plot(z_cent, integrated, lw=1.4, label=label)

ax.set_xlabel("stage z (arb.)")
ax.set_ylabel(
    "Σ (D - baseline) / G over ROI" if sym_roi
    else "Σ D / G over ROI  (counts/shot/uJ)"
)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("eTOF integrated over ROI vs delay")
plt.show()

## Plot 4 — partial covariance in a delay window

For each file, average the per-(z) per-shot first and second moments
over `COV_DELAY_WINDOW` (shot-weighted by `n_per_bin`), downbin via
the matrix transformation (`M @ X @ Mᵀ` for second moments, `x @ Mᵀ`
for first moments), then form:

  * **Electron-ion**:
    `Cov(D, C | G) = Cov(D, C) − α · Cov(D, G) Cov(G, C) / Var(G)`
  * **Ion-ion**:
    `Cov(C, C | G) = Cov(C, C) − α · Cov(C, G) Cov(C, G)ᵀ / Var(G)`

with `α = OVERSUBTRACT`. `α > 1` over-subtracts the GMD common mode
to suppress residual fluence correlation at the cost of biasing
true correlated signal.

Top metric row per file: `Cov(D, C | G)` (rows = eTOF, cols = ion
TOF). Bottom metric row: `Cov(C, C | G)` (both axes = ion TOF).
Files wrap to multiple subplot-rows past `MAX_COLS` columns.

In [ ]:
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(2, n_files,
                                           panel_size=(6, 5),
                                           sharey=False)

for k, (label, run) in enumerate(runs.items()):
    agg = run["agg"]
    M_e = run["M_e"]
    M_i = run["M_i"]
    n_per_bin_z = agg.n_per_bin[GMD_BIN]                # (n_z,)

    # Resolve the delay-window indices on the original z axis.
    lo, hi = COV_DELAY_WINDOW
    if lo is None and hi is None:
        j0, j1 = 0, run["n_z"]
    else:
        j0, j1 = _window_indices(agg.z_edges, lo, hi)

    # Shot-weighted means over the delay window (un-downbinned axes).
    D_m   = _window_avg(agg.D[GMD_BIN],   n_per_bin_z, j0, j1)
    C_m   = _window_avg(agg.C[GMD_BIN],   n_per_bin_z, j0, j1)
    G_m   = _window_avg(agg.G[GMD_BIN],   n_per_bin_z, j0, j1)
    GtG_m = _window_avg(agg.GtG[GMD_BIN], n_per_bin_z, j0, j1)
    DtD_m = _window_avg(agg.DtD[GMD_BIN], n_per_bin_z, j0, j1)
    CtC_m = _window_avg(agg.CtC[GMD_BIN], n_per_bin_z, j0, j1)
    DtC_m = _window_avg(agg.DtC[GMD_BIN], n_per_bin_z, j0, j1)
    DtG_m = _window_avg(agg.DtG[GMD_BIN], n_per_bin_z, j0, j1)
    CtG_m = _window_avg(agg.CtG[GMD_BIN], n_per_bin_z, j0, j1)

    # Apply the downbinning matrix transformation.
    D_m   = M_e @ D_m
    C_m   = M_i @ C_m
    DtD_m = M_e @ DtD_m @ M_e.T
    CtC_m = M_i @ CtC_m @ M_i.T
    DtC_m = M_e @ DtC_m @ M_i.T
    DtG_m = M_e @ DtG_m
    CtG_m = M_i @ CtG_m

    # Covariances.
    cov_DC = DtC_m - D_m[:, None] * C_m[None, :]
    cov_CC = CtC_m - C_m[:, None] * C_m[None, :]
    cov_DG = DtG_m - D_m * G_m
    cov_CG = CtG_m - C_m * G_m
    var_G  = float(GtG_m - G_m ** 2)
    safe_var = var_G if var_G > 0 else np.nan

    # Partial covariances with the OVERSUBTRACT factor.
    correction_DC  = (cov_DG[:, None] * cov_CG[None, :]) / safe_var
    correction_CC  = (cov_CG[:, None] * cov_CG[None, :]) / safe_var
    cov_DC_partial = cov_DC - OVERSUBTRACT * correction_DC
    cov_CC_partial = cov_CC - OVERSUBTRACT * correction_CC

    n_shots = int(n_per_bin_z[j0:j1].sum())
    win_tag = (f"z [{agg.z_edges[j0]:.0f}, {agg.z_edges[j1]:.0f})"
               f"  n={n_shots}  alpha={OVERSUBTRACT:.2f}")

    # Plot electron-ion partial cov.
    extent_DC = [run["ion_tof_edges"][0], run["ion_tof_edges"][-1],
                 run["tof_edges"][0],     run["tof_edges"][-1]]
    vmax = float(np.nanpercentile(np.abs(cov_DC_partial), PCT_HIGH)) or 1.0
    ax_dc = ax_at(k, 0)
    im = ax_dc.imshow(cov_DC_partial, origin="lower", cmap="RdBu_r",
                      vmin=-vmax, vmax=vmax, extent=extent_DC,
                      aspect="auto")
    ax_dc.set_title(f"{label}: Cov(D, C | G)\n{win_tag}", fontsize=10)
    ax_dc.set_xlabel("ion TOF (100 ps)")
    ax_dc.set_ylabel("eTOF (100 ps)")
    fig.colorbar(im, ax=ax_dc, fraction=0.046)

    # Plot ion-ion partial cov.
    extent_CC = [run["ion_tof_edges"][0], run["ion_tof_edges"][-1],
                 run["ion_tof_edges"][0], run["ion_tof_edges"][-1]]
    vmax = float(np.nanpercentile(np.abs(cov_CC_partial), PCT_HIGH)) or 1.0
    ax_cc = ax_at(k, 1)
    im = ax_cc.imshow(cov_CC_partial, origin="lower", cmap="RdBu_r",
                      vmin=-vmax, vmax=vmax, extent=extent_CC,
                      aspect="auto")
    ax_cc.set_title(f"{label}: Cov(C, C | G)\n{win_tag}", fontsize=10)
    ax_cc.set_xlabel("ion TOF (100 ps)")
    ax_cc.set_ylabel("ion TOF (100 ps)")
    fig.colorbar(im, ax=ax_cc, fraction=0.046)

hide_unused()
fig.suptitle(f"Partial covariance over the delay window  "
             f"(GMD bin {GMD_BIN}, oversubtract = {OVERSUBTRACT:.2f})")
plt.show()